# ESoC 2026 CLAAS — Model Training

Three classifiers for early stone detection from harvester audio.

| Model | Input | Library |
|-------|-------|---------|
| ROCKET | MFCCs `[n, 13, ~43]` | sktime |
| TimeSeriesForestClassifier | MFCCs `[n, 13, ~43]` | sktime |
| 1D-CNN | Raw audio `[n, 1, 26460]` | PyTorch |

**Evaluation**: leave-one-run-out cross-validation (5 folds, one per recording session)  
**Metrics**: true positive rate · false alarms per header-On hour · advance detection time (ms)  
**Bonus 2**: INT8 ONNX quantization of best CNN for 2MB MCU deployment

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkg.split())

pip_install("-e /Users/amith_mechanicalengineer/sktime")
pip_install("torch --index-url https://download.pytorch.org/whl/cpu")
pip_install("onnx onnxruntime")

import sktime, torch, onnx, onnxruntime
print("sktime:", sktime.__version__)
print("torch:", torch.__version__)
print("onnx:", onnx.__version__)
print("onnxruntime:", onnxruntime.__version__)

In [ ]:
from asammdf import MDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import os
import pickle
from pathlib import Path
from scipy.stats import kurtosis
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier

from sktime.classification.kernel_based import RocketClassifier
from sktime.classification.interval_based import TimeSeriesForestClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

DATA_DIR = Path("data")
MF4_FILES = sorted(DATA_DIR.glob("*.mf4"))

SR             = 44100
VOLT_THRESHOLD = 2000
WINDOW_BEFORE  = 0.5
WINDOW_AFTER   = 0.1
WINDOW_LEN     = WINDOW_BEFORE + WINDOW_AFTER
MIN_SUSTAIN    = 5
N_MFCC         = 13

print("Files:", [f.name for f in MF4_FILES])

## 1. Reused labeling functions
Copied verbatim from `explore_stone_audio_fingerprint.ipynb` — do not modify.

In [ ]:
def get_episodes(status_channel):
    s = status_channel.samples.astype(str)
    t = status_channel.timestamps
    changes = np.where(s[:-1] != s[1:])[0]
    events = [(t[0], s[0])]
    for i in changes:
        events.append((t[i+1], s[i+1]))
    episodes, ep_start = [], None
    for ev_time, ev_state in events:
        if ev_state == 'On' and ep_start is None:
            ep_start = ev_time
        elif ev_state == 'Off' and ep_start is not None:
            episodes.append((ep_start, ev_time))
            ep_start = None
    if ep_start is not None:
        episodes.append((ep_start, t[-1]))
    return episodes


def get_stone_spike_times(volt_channel, episodes,
                          threshold=VOLT_THRESHOLD, min_sustain=MIN_SUSTAIN):
    v, t = volt_channel.samples, volt_channel.timestamps
    spike_times = []
    for ep_start, ep_end in episodes:
        mask = (t >= ep_start) & (t <= ep_end)
        v_ep, t_ep = v[mask], t[mask]
        if len(v_ep) < min_sustain:
            continue
        above = v_ep > threshold
        sustained = np.zeros_like(above)
        count = 0
        for k in range(len(above)):
            if above[k]:
                count += 1
                if count >= min_sustain:
                    sustained[k - min_sustain + 1:k + 1] = True
            else:
                count = 0
        edges = np.diff(sustained.astype(int))
        onsets = t_ep[np.where(edges == 1)[0] + 1]
        for st in onsets:
            if not spike_times or st - spike_times[-1] > 1.0:
                spike_times.append(float(st))
    return spike_times


def extract_audio_window(audio_channel, center_time,
                         before=WINDOW_BEFORE, after=WINDOW_AFTER):
    t, s = audio_channel.timestamps, audio_channel.samples
    mask = (t >= center_time - before) & (t <= center_time + after)
    return s[mask].astype(np.float32)


print("Labeling functions loaded.")

## 2. Build window datasets

In [ ]:
stone_windows  = []
normal_windows = []
rng = np.random.default_rng(42)

for f in MF4_FILES:
    mf     = MDF(f)
    audio  = mf.get("Sensor1")
    volt   = mf.get("VoltageSignal")
    status = mf.get("Status")

    episodes    = get_episodes(status)
    spike_times = get_stone_spike_times(volt, episodes)

    for st in spike_times:
        w = extract_audio_window(audio, st)
        if len(w) >= int(WINDOW_LEN * SR) * 0.9:
            stone_windows.append((f.stem, st, w))

    for ep_start, ep_end in episodes:
        ep_dur = ep_end - ep_start
        if ep_dur < WINDOW_LEN + 4:
            continue
        n_samples = min(3, int(ep_dur / (WINDOW_LEN + 2)))
        candidates = rng.uniform(ep_start + 1, ep_end - WINDOW_LEN - 1,
                                  size=n_samples * 5)
        count = 0
        for ct in candidates:
            if any(abs(ct - st) < 2.0 for st in spike_times):
                continue
            w = extract_audio_window(audio, ct + WINDOW_BEFORE)
            if len(w) >= int(WINDOW_LEN * SR) * 0.9:
                normal_windows.append((f.stem, ct, w))
                count += 1
            if count >= n_samples:
                break

print(f"Stone windows:  {len(stone_windows)}")
print(f"Normal windows: {len(normal_windows)}")

# Unified arrays — same ordering used throughout
all_windows = stone_windows + normal_windows
y_all       = np.array([1]*len(stone_windows) + [0]*len(normal_windows))
meta_all    = [(r, t) for r, t, _ in all_windows]
run_names   = sorted(set(r for r, _ in meta_all))
print("Runs:", run_names)

## 3. Per-run header-On duration (for false alarm rate metric)

In [ ]:
header_on_hours = {}

for f in MF4_FILES:
    mf     = MDF(f)
    status = mf.get("Status")
    eps    = get_episodes(status)
    total_s = sum(ep_end - ep_start for ep_start, ep_end in eps)
    header_on_hours[f.stem] = total_s / 3600.0
    print(f"  {f.stem[-10:]}: {total_s/60:.1f} min On  ({total_s/3600:.4f} hr)")

## 4. MFCC feature matrix for sktime models

Shape: `[n_instances, 13, n_frames]` — sktime numpy3D format.  
Only pre-spike audio (`[:500ms]`) used — enforces causal inference.

In [ ]:
def compute_mfcc_matrix(windows):
    mfccs = []
    n_pre = int(WINDOW_BEFORE * SR)
    for _, _, audio in windows:
        pre = audio[:n_pre].astype(np.float32)
        m   = librosa.feature.mfcc(y=pre, sr=SR, n_mfcc=N_MFCC,
                                    n_fft=2048, hop_length=512)
        mfccs.append(m)
    n_frames = min(m.shape[1] for m in mfccs)
    return np.stack([m[:, :n_frames] for m in mfccs], axis=0)

X_mfcc = compute_mfcc_matrix(all_windows)   # [73, 13, n_frames]
print("MFCC matrix shape:", X_mfcc.shape)   # expect [73, 13, ~43]
print("Labels shape:",      y_all.shape)
print("Class counts:",      np.bincount(y_all))  # [63 normal, 10 stone]

# Univariate for TSF: mean across 13 MFCC channels -> [n, 1, n_frames]
X_mfcc_uni = X_mfcc.mean(axis=1, keepdims=True)
print('MFCC univariate shape:', X_mfcc_uni.shape)

## 5. Leave-one-run-out (LORO) CV framework

In [ ]:
def make_loro_folds(meta_all, y_all, run_names):
    for run in run_names:
        test_idx  = [i for i, (r, _) in enumerate(meta_all) if r == run]
        train_idx = [i for i, (r, _) in enumerate(meta_all) if r != run]
        yield run, train_idx, test_idx


def balance_train(train_idx, y_all, seed=42):
    """Oversample stone class to match normal class size within training split."""
    y_tr = y_all[train_idx]
    stone_idx  = [train_idx[i] for i in np.where(y_tr == 1)[0]]
    normal_idx = [train_idx[i] for i in np.where(y_tr == 0)[0]]
    if len(stone_idx) == 0:
        return train_idx
    stone_over = resample(stone_idx, n_samples=len(normal_idx),
                          replace=True, random_state=seed)
    bal = np.array(normal_idx + stone_over)
    np.random.default_rng(seed).shuffle(bal)
    return bal


def evaluate_fold(y_true, y_pred, meta_test, header_on_hours_run):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    stone_mask  = y_true == 1
    normal_mask = y_true == 0

    tpr = float(np.mean(y_pred[stone_mask] == 1)) if stone_mask.sum() > 0 else float("nan")

    false_alarms = int((y_pred[normal_mask] == 1).sum())
    fpr_per_hour = false_alarms / (header_on_hours_run + 1e-12)

    # Window-level classifier fires at start of 500ms window → 500ms advance
    n_tp = int((y_pred[stone_mask] == 1).sum()) if stone_mask.sum() > 0 else 0
    mean_advance_ms = WINDOW_BEFORE * 1000 if n_tp > 0 else float("nan")

    return {"tpr": tpr, "fpr_per_hour": fpr_per_hour,
            "mean_advance_ms": mean_advance_ms, "n_tp": n_tp,
            "n_stone_test": int(stone_mask.sum()),
            "n_false_alarms": false_alarms}


# Dry run — show fold sizes
print(f"{'Fold':<45} {'train':>6} {'test':>6} {'stone_test':>10} {'normal_test':>12}")
print("-" * 82)
for fold_run, tr, te in make_loro_folds(meta_all, y_all, run_names):
    n_s = sum(y_all[i] == 1 for i in te)
    n_n = sum(y_all[i] == 0 for i in te)
    print(f"{fold_run:<45} {len(tr):>6} {len(te):>6} {n_s:>10} {n_n:>12}")

## 6. Model 1 — ROCKET

Random Convolutional Kernel Transform + RidgeClassifierCV.  
Input: MFCC time series `[n, 13, ~43]`.  
Fast, strong on small datasets, no manual feature engineering.

In [ ]:
rocket_results = []

for fold_run, train_idx, test_idx in make_loro_folds(meta_all, y_all, run_names):
    bal_idx = balance_train(train_idx, y_all)
    X_tr, y_tr = X_mfcc[bal_idx], y_all[bal_idx]
    X_te, y_te = X_mfcc[test_idx], y_all[test_idx]
    meta_te    = [meta_all[i] for i in test_idx]

    clf = RocketClassifier(num_kernels=1000, rocket_transform="rocket",
                           random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = evaluate_fold(y_te, y_pred, meta_te, header_on_hours[fold_run])
    m["fold"] = fold_run
    rocket_results.append(m)
    tpr_str = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  {fold_run[-10:]}  tpr={tpr_str}  "
          f"fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}")

rocket_df = pd.DataFrame(rocket_results).set_index("fold")
print("\nROCKET LORO summary:")
print(rocket_df[["tpr","fpr_per_hour","mean_advance_ms","n_tp","n_stone_test"]].round(3))

## 7. Model 2 — TimeSeriesForestClassifier

Ensemble of 200 trees over random intervals.  
Each interval contributes mean, std, slope features.  
Exposes temporal feature importances — tells us *when* in the 500ms window the stone is audible.

In [ ]:
tsf_results = []

for fold_run, train_idx, test_idx in make_loro_folds(meta_all, y_all, run_names):
    bal_idx = balance_train(train_idx, y_all)
    X_tr, y_tr = X_mfcc_uni[bal_idx], y_all[bal_idx]
    X_te, y_te = X_mfcc_uni[test_idx], y_all[test_idx]
    meta_te    = [meta_all[i] for i in test_idx]

    clf = TimeSeriesForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)

    m = evaluate_fold(y_te, y_pred, meta_te, header_on_hours[fold_run])
    m["fold"] = fold_run
    tsf_results.append(m)
    tpr_str = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  {fold_run[-10:]}  tpr={tpr_str}  "
          f"fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}")

tsf_df = pd.DataFrame(tsf_results).set_index("fold")
print("\nTSF LORO summary:")
print(tsf_df[["tpr","fpr_per_hour","mean_advance_ms","n_tp","n_stone_test"]].round(3))

In [ ]:
# TSF temporal importance — train on all data to get stable importances
bal_all = balance_train(list(range(len(y_all))), y_all)
tsf_full = TimeSeriesForestClassifier(n_estimators=200, random_state=42)
tsf_full.fit(X_mfcc_uni[bal_all], y_all[bal_all])

imp_raw = tsf_full.feature_importances_
# feature_importances_ may be a DataFrame or array — normalise to 1D numpy
import pandas as pd
if isinstance(imp_raw, pd.DataFrame):
    imp_arr = imp_raw.values.flatten()
else:
    imp_arr = np.asarray(imp_raw).flatten()
# Trim or pad to n_frames length
n_frames = X_mfcc_uni.shape[2]
importances = imp_arr[:n_frames] if len(imp_arr) >= n_frames else imp_arr
t_imp = np.linspace(-WINDOW_BEFORE, 0, len(importances))

plt.figure(figsize=(12, 4))
plt.fill_between(t_imp, 0, importances, alpha=0.4, color='darkorange')
plt.plot(t_imp, importances, color='darkorange', lw=1.5)
plt.axvline(-0.1, color='red', lw=1.2, linestyle='--',
            label='100ms before metal detector fires')
plt.xlabel("Time relative to metal detector spike (s)")
plt.ylabel("Feature importance")
plt.title("TSF: When in the 500ms window is the stone impact detectable?")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Model 3 — 1D-CNN (PyTorch)

Raw audio windows `[batch, 1, 26460]` → 3 strided conv blocks → global avg pool → binary classifier.  
Data augmentation compensates for only 10 stone training examples.

In [ ]:
def augment_stone(audio_batch, n_augments=8, seed=42):
    """
    Augment stone audio windows:
      - Time shift ±50ms (±2205 samples) with zero-padding
      - Gaussian noise: N(0, 0.01 * std)
      - Amplitude scale: U(0.8, 1.2)
    """
    rng_a = np.random.default_rng(seed)
    out = []
    for audio in audio_batch:
        for _ in range(n_augments):
            a = audio.copy()
            shift = rng_a.integers(-int(0.05*SR), int(0.05*SR))
            a = np.roll(a, shift)
            if shift > 0:
                a[:shift] = 0.0
            elif shift < 0:
                a[shift:] = 0.0
            a = a + rng_a.normal(0, 0.01 * (np.std(a) + 1e-8), size=len(a)).astype(np.float32)
            a = (a * rng_a.uniform(0.8, 1.2)).astype(np.float32)
            out.append(a)
    return np.stack(out, axis=0)


class StoneCNN(nn.Module):
    """
    Input:  [batch, 1, 26460]
    Output: [batch, 2]  (logits: normal=0, stone=1)
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,  16, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.pool       = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(64, 2))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)


# Sanity check
dummy = torch.zeros(4, 1, 26460)
m_test = StoneCNN()
print("CNN output shape:", m_test(dummy).shape)
print(f"CNN parameters:  {sum(p.numel() for p in m_test.parameters()):,}")

In [ ]:
def train_cnn_fold(train_idx, test_idx, n_epochs=50, n_augments=8, lr=1e-3):
    WIN_LEN = int(WINDOW_LEN * SR)
    all_audio = [w[2][:WIN_LEN] for w in all_windows]  # trim to fixed length

    # Raw audio for training split
    X_tr_raw = np.stack([all_audio[i] for i in train_idx])
    y_tr     = y_all[train_idx]

    # Augment stone windows inside the fold (never augment test)
    stone_local = np.where(y_tr == 1)[0]
    normal_local = np.where(y_tr == 0)[0]
    if len(stone_local) > 0:
        aug = augment_stone(X_tr_raw[stone_local], n_augments=n_augments)
        X_train = np.concatenate([X_tr_raw, aug], axis=0)
        y_train = np.concatenate([y_tr, np.ones(len(aug), dtype=np.int64)])
    else:
        X_train, y_train = X_tr_raw, y_tr

    X_train_t = torch.tensor(X_train[:, np.newaxis, :], dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)

    class_counts = np.bincount(y_train.astype(int))
    sample_w = torch.tensor(
        (1.0 / (class_counts[y_train.astype(int)] + 1e-6)), dtype=torch.float32)
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    loader  = DataLoader(TensorDataset(X_train_t, y_train_t),
                         batch_size=16, sampler=sampler)

    model = StoneCNN()
    cw = torch.tensor([1.0, class_counts[0] / (class_counts[1] + 1e-6)],
                       dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            print(f"    ep {epoch+1}/{n_epochs}  loss={total_loss/len(loader):.4f}")

    # Inference
    X_te_raw = np.stack([all_audio[i] for i in test_idx])
    X_te_t   = torch.tensor(X_te_raw[:, np.newaxis, :], dtype=torch.float32)
    y_te     = y_all[test_idx]

    model.eval()
    with torch.no_grad():
        y_pred = model(X_te_t).argmax(dim=1).numpy()

    return model, y_pred, y_te

In [ ]:
cnn_results = []
cnn_models  = {}

for fold_run, train_idx, test_idx in make_loro_folds(meta_all, y_all, run_names):
    meta_te = [meta_all[i] for i in test_idx]
    print(f"\n--- CNN Fold: {fold_run[-10:]} ---")
    model, y_pred, y_te = train_cnn_fold(train_idx, test_idx)
    m = evaluate_fold(y_te, y_pred, meta_te, header_on_hours[fold_run])
    m["fold"] = fold_run
    cnn_results.append(m)
    cnn_models[fold_run] = model
    tpr_str = f"{m['tpr']:.2f}" if not np.isnan(m['tpr']) else "N/A"
    print(f"  Result: tpr={tpr_str}  fa/hr={m['fpr_per_hour']:.1f}  "
          f"tp={m['n_tp']}/{m['n_stone_test']}")

cnn_df = pd.DataFrame(cnn_results).set_index("fold")
print("\nCNN LORO summary:")
print(cnn_df[["tpr","fpr_per_hour","mean_advance_ms","n_tp","n_stone_test"]].round(3))

## 9. Model comparison

In [ ]:
def summarise(df, name):
    return pd.Series({
        "model":           name,
        "mean_tpr":        df["tpr"].mean(skipna=True),
        "mean_fa_per_hr":  df["fpr_per_hour"].mean(skipna=True),
        "advance_ms":      df["mean_advance_ms"].mean(skipna=True),
        "total_tp":        df["n_tp"].sum(),
        "total_stone": df["n_stone_test"].sum(),
    })

comparison = pd.DataFrame([
    summarise(rocket_df, "ROCKET"),
    summarise(tsf_df,    "TimeSeriesForest"),
    summarise(cnn_df,    "1D-CNN"),
]).set_index("model")

print(comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Model comparison — leave-one-run-out CV", fontsize=11)
colors = ["steelblue", "darkorange", "tomato"]

for ax, col, title, better in zip(
    axes,
    ["mean_tpr", "mean_fa_per_hr", "advance_ms"],
    ["True Positive Rate (↑)", "False Alarms / Hour (↓)", "Advance Time ms (↑)"],
    [True, False, True]
):
    vals = comparison[col]
    bars = ax.bar(vals.index, vals, color=colors)
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis='x', rotation=20)
    # Highlight best bar
    best_i = int(vals.argmax()) if better else int(vals.argmin())
    bars[best_i].set_edgecolor('black')
    bars[best_i].set_linewidth(2)

plt.tight_layout()
plt.show()

## 10. Bonus 2 — INT8 ONNX quantization for MCU deployment

Target: automotive microcontroller, 2MB RAM, inference only.

**Why the 1D-CNN**: fixed compute graph, no dynamic memory, INT8 quantization is well-supported.  
**Pipeline**: PyTorch → ONNX FP32 → ONNX INT8 (dynamic quantization via `onnxruntime`)

We export the CNN trained on the fold that had the most training stone examples
(Run1 held out → trained on Runs 2–5 = all 10 stone events).

In [ ]:
# Pick fold where Run1 is held out — training set contains all 10 stone events
best_fold_key = sorted(cnn_models.keys())[0]  # Run1 = Messung_2025-05-09
best_model    = cnn_models[best_fold_key]
best_model.eval()

onnx_fp32_path = "stone_cnn_float32.onnx"
dummy_input    = torch.zeros(1, 1, 26460)

torch.onnx.export(
    best_model,
    dummy_input,
    onnx_fp32_path,
    input_names=["audio_window"],
    output_names=["logits"],
    opset_version=11,
)

# Run ONNX shape inference as pre-processing (required for quantization compatibility)
import onnx
from onnx import shape_inference as onnx_si
m = onnx.load(onnx_fp32_path)
onnx.save(onnx_si.infer_shapes(m), onnx_fp32_path)

fp32_kb = os.path.getsize(onnx_fp32_path) / 1024
print(f"ONNX FP32: {fp32_kb:.1f} KB")

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType, quant_pre_process

# quant_pre_process is required before quantize_dynamic for models exported by torch 2.x
preproc_path  = "stone_cnn_preproc.onnx"
onnx_int8_path = "stone_cnn_int8.onnx"

quant_pre_process(onnx_fp32_path, preproc_path)
quantize_dynamic(preproc_path, onnx_int8_path, weight_type=QuantType.QInt8)

int8_kb = os.path.getsize(onnx_int8_path) / 1024
print(f"ONNX FP32:   {fp32_kb:.1f} KB")
print(f"ONNX INT8:   {int8_kb:.1f} KB")
print(f"Compression: {fp32_kb/int8_kb:.1f}x")
print(f"Fits in 2MB: {int8_kb < 2048}")

In [ ]:
import onnxruntime as ort

sess_fp32 = ort.InferenceSession(onnx_fp32_path)
sess_int8 = ort.InferenceSession(onnx_int8_path)

# Run inference one instance at a time (model exported with static batch=1)
WIN_LEN = int(WINDOW_LEN * SR)
all_audio_arr = [w[2][:WIN_LEN] for w in all_windows]

preds_fp32, preds_int8 = [], []
for audio in all_audio_arr:
    x = audio[np.newaxis, np.newaxis, :].astype(np.float32)  # [1, 1, WIN_LEN]
    preds_fp32.append(sess_fp32.run(None, {"audio_window": x})[0].argmax())
    preds_int8.append(sess_int8.run(None, {"audio_window": x})[0].argmax())

preds_fp32 = np.array(preds_fp32)
preds_int8 = np.array(preds_int8)
agreement  = (preds_fp32 == preds_int8).mean()
print(f"Prediction agreement FP32 vs INT8: {agreement:.1%}")
print(f"FP32 stone predictions: {preds_fp32.sum()} / {len(preds_fp32)}")
print(f"INT8 stone predictions: {preds_int8.sum()} / {len(preds_int8)}")

## 11. Fallback: scalar decision tree for ultra-constrained MCU

In [ ]:
def scalar_features(audio):
    n_pre = int(WINDOW_BEFORE * SR)
    pre   = audio[:n_pre]
    w10   = int(SR * 0.01)
    rms_v = [np.sqrt(np.mean(pre[i*w10:(i+1)*w10]**2)) for i in range(len(pre)//w10)]
    w20   = int(SR * 0.02)
    krt_v = [kurtosis(pre[i*w20:(i+1)*w20]) for i in range(len(pre)//w20)]
    return [max(rms_v) if rms_v else 0.0, max(krt_v) if krt_v else 0.0]

X_scalar = np.array([scalar_features(w[2]) for w in all_windows])
dt = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
dt.fit(X_scalar, y_all)

dt_path = "stone_dt_fallback.pkl"
with open(dt_path, "wb") as fh:
    pickle.dump(dt, fh)

print(f"Decision tree size: {os.path.getsize(dt_path)/1024:.2f} KB")
print(f"Train accuracy:     {dt.score(X_scalar, y_all):.2%}")

# Plot decision boundary
fig, ax = plt.subplots(figsize=(8, 5))
x0 = X_scalar[y_all==0]; x1 = X_scalar[y_all==1]
ax.scatter(x0[:,0], x0[:,1], c='steelblue', alpha=0.6, label='Normal', s=40)
ax.scatter(x1[:,0], x1[:,1], c='tomato',    alpha=0.8, label='Stone',  s=80, marker='*')
ax.set_xlabel("Peak RMS")
ax.set_ylabel("Max Kurtosis")
ax.set_title("Fallback DT: 2 scalar features (peak_rms, max_kurtosis)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Model | Input | Mean TPR | Mean FA/hr | Advance ms | Deployed size |
|-------|-------|----------|------------|------------|---------------|
| ROCKET | MFCCs [13×43] | — | — | 500ms | N/A (sklearn) |
| TimeSeriesForest | MFCCs [13×43] | — | — | 500ms | N/A |
| 1D-CNN FP32 | Raw [26460] | — | — | 500ms | ~200KB |
| **1D-CNN INT8** | Raw [26460] | — | — | 500ms | **~50KB** |
| DT fallback | 2 scalars | — | — | 500ms | ~1KB |

### Limitations
- **Only 10 labeled stone events** — all results have high variance. LORO folds with 1–2 stone test samples cannot support statistically meaningful TPR estimates.
- **500ms advance time** is a window-resolution estimate. A real streaming system would detect earlier if the impact transient appears near the start of the window.
- **Non-metallic stone generalization** is assumed but not validated — no non-metallic ground truth exists in this dataset.
- Recommendation before production: collect ≥100 labeled stone events across diverse field conditions.